# 동적 페이지 데이터를 수집하자

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🌐 자바스크립트 너머의 데이터 — 동적 페이지 수집과 수집 윤리

### — 화면엔 보이는데 소스엔 없는 데이터를, 책임 있게 가져오는 법 —

---

## 📋 학습 목표

이 자습서를 마치면 여러분은 다음을 할 수 있습니다.

1. JavaScript 렌더링 페이지가 왜 정적 수집으로 안 잡히는지, 그 원리를 설명할 수 있습니다.
2. Playwright 같은 브라우저 자동화 도구가 어떻게 동적 페이지를 수집하는지 설명할 수 있습니다.
3. AI 보조 코딩 4단계(목적 → 생성 → 검증 → 수정)로 수집 코드를 만들고 **검증**할 수 있습니다.
4. `robots.txt`를 확인하고, 수집해도 되는 범위인지 스스로 판단할 수 있습니다.
5. 요청 속도 제한·User-Agent 등 "서버에 부담을 주지 않는" 예의를 코드로 지킬 수 있습니다.
6. 수집한 데이터를 분석 가능한 DataFrame으로 정리하고, 앞서 배운 랭글링 기술을 다시 적용할 수 있습니다.
7. 코드를 짜기 전에 **페이지 유형과 허용 범위를 먼저 판단하는** 습관을 들일 수 있습니다.

> 💡 위 목록을 천천히 읽고, 지금 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해보세요. 오늘은 도구를 늘리는 날인 동시에, **"가져와도 되는가"를 먼저 묻는 태도**를 기르는 날입니다.

## 📚 목차

| Part | 내용 | 핵심 질문 |
| --- | --- | --- |
| Part 0 | 분석 상황과 학습 지도 | 어제 정적 수집이 막혔던 그 페이지, 오늘 어떻게 뚫을까요? |
| Part 1 | 정적 수집의 실패 재확인 | 분명 화면엔 보이는데, 왜 결과가 비었을까요? |
| Part 2 | JavaScript 렌더링 원리 | 서버가 준 HTML과 브라우저가 보여주는 화면은 왜 다를까요? |
| Part 3 | Playwright 동작·설치 | 브라우저를 코드로 조종하면 무엇이 달라질까요? |
| Part 4 | AI 보조 코딩 4단계 | AI가 짜준 수집 코드, 그대로 믿어도 될까요? |
| Part 5 | 동적 페이지 상호작용 | 로그인·무한스크롤·팝업은 어떻게 다룰까요? |
| Part 6 | 수집 윤리·법·안전 | 가져와도 되는지 어떻게 판단할까요? |
| 종합 실습 | 동적 수집 + 윤리 검토 리포트 | 새 페이지를 처음부터 끝까지 책임 있게 수집할 수 있을까요? |
| 정리 | 핵심 요약 · 다음 시간 | 수집·정제한 데이터로, 이제 무엇을 말할까요? |

## 분석 상황과 학습 지도

# 0. 분석 상황과 학습 지도

## 지난 시간에는 무엇을 했나요?

바로 전 시간(D+010)에는 **웹에서 데이터를 꺼내는 첫걸음**을 뗐습니다. HTTP 요청·응답 구조를 보고, `requests`로 웹 페이지의 HTML을 받아온 뒤, **BeautifulSoup**으로 원하는 데이터를 추출했죠. 정적 페이지(static page) — 즉 **HTML 소스 안에 데이터가 그대로 들어 있는** 페이지 — 는 이 방법으로 깔끔하게 수집됐습니다.

그런데 마지막에 이상한 일이 있었습니다. 어떤 페이지는 **브라우저 화면엔 상품이 분명히 보이는데**, `requests`로 받은 HTML에는 상품이 **하나도 없었습니다.** BeautifulSoup이 빈 결과를 돌려줬죠. "분명 보이는데 왜 못 가져오지?" — 오늘은 바로 그 미스터리에서 출발합니다.

## 오늘의 여정

오늘은 **동적 페이지(dynamic page)** — JavaScript가 화면을 그리는 페이지 — 를 수집합니다. 원리를 이해하고, 브라우저를 코드로 조종하는 Playwright를 만나고, AI의 도움을 검증하며 쓰는 법, 그리고 무엇보다 **"가져와도 되는가"를 판단하는 윤리**까지 배웁니다.

```text
[Part 1] 정적의 실패 재확인   →  왜 결과가 비었나 (소스엔 데이터가 없다)
   ↓
[Part 2] JS 렌더링 원리       →  서버가 준 HTML ≠ 브라우저가 보여주는 화면
   ↓
[Part 3] Playwright           →  브라우저를 코드로 조종해 '렌더링된 DOM'을 얻는다
   ↓
[Part 4] AI 보조 코딩 4단계    →  목적→생성→검증→수정 (검증이 핵심!)
   ↓
[Part 5] 동적 상호작용         →  로그인·무한스크롤·팝업 (샌드박스 한정)
   ↓
[Part 6] 수집 윤리·법         →  robots.txt·요청 예의·저작권·개인정보
   ↓
[종합 실습] 새 페이지 책임 있게 수집 + 윤리 리포트
```

## 이 자습서 사용법

이 노트북은 **혼자서도 끝까지 학습할 수 있도록** 만들었습니다. 다음 네 박자로 진행하세요.

- 📖 **읽고** — 개념과 도식을 천천히 읽습니다.
- 💻 **실행하고** — 코드 셀을 위에서부터 순서대로 실행합니다. (단축키: `Shift + Enter`)
- ✏️ **고쳐보고** — "스스로 해보자!" 칸에서 코드를 직접 바꿔봅니다.
- 🤔 **답해보고** — 체크포인트 질문에 스스로 답해봅니다. 틀려도 괜찮습니다.

> ⚠ **이 노트북의 특별한 점 — 네트워크 없이도 끝까지 돌아갑니다.** 웹 스크래핑은 "살아 있는 웹"을 다루기 때문에, 사이트가 막히거나 브라우저가 없으면 코드가 깨질 수 있어요. 그래서 이 노트북은 **핵심 개념을 모두 '노트북 안에 넣어둔 HTML'로** 가르칩니다. 네트워크나 브라우저가 필요한 셀은 `# [네트워크 필요]`·`# [브라우저 설치 필요]` 주석을 달고, 안 되면 **자동으로 내장 데이터로 폴백**하도록 만들었습니다. 그러니 안심하고 위에서부터 실행하세요.

## 오늘의 무대: 다시, 가상의 이커머스 "모두마켓"

이번 모듈 내내 우리는 가상의 온라인 쇼핑몰 **모두마켓**의 데이터를 다뤄 왔습니다. 그런데 이번엔 데이터가 *파일로 주어지지 않았습니다.* 모두마켓의 **신상품 페이지를 직접 수집**해야 합니다.

문제는, 이 신상품 페이지가 **동적 페이지**라는 점입니다. 즉 서버는 빈 껍데기 HTML만 주고, 실제 상품 목록은 **브라우저가 JavaScript를 실행한 뒤에야** 화면에 그려집니다.

아래 셀을 실행하면 두 가지 버전의 HTML이 만들어집니다. 둘을 비교하는 것이 오늘 개념의 출발점이에요.

- `STATIC_HTML` — **서버가 보내준 그대로**의 HTML (= `requests`가 받는 것). 상품 데이터가 **없습니다.**
- `RENDERED_HTML` — **브라우저가 JS를 실행해 상품을 다 그린 뒤**의 HTML (= 우리가 화면에서 보는 것). 상품 데이터가 **있습니다.**

> **🎯 오늘의 핵심**
> **도구를 고르기 전에 페이지 유형과 허용 범위부터 판단합니다.** "정적인가 동적인가", "가져와도 되는가"를 먼저 묻는 것이 수집의 첫 행동입니다.

> **읽는 법:** `STATIC_HTML`의 `#product-list` 안에는 `상품을 불러오는 중...`이라는 안내문만 있고, 진짜 상품은 `<script>`가 나중에 그립니다. 반면 `RENDERED_HTML`에는 `<div class="product">`가 8개 들어 있죠. **같은 페이지인데 '언제의 HTML이냐'에 따라 내용이 완전히 다릅니다.** 이 차이가 오늘의 모든 것입니다.

이제 어제(D+010) 막혔던 장면을 코드로 재현하며, 왜 비었는지부터 확인해봅시다.

In [ ]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 점검 (없어도 노트북은 폴백으로 끝까지 진행)
# ─────────────────────────────────────────────
# 설치가 필요하면 아래 주석을 해제하세요.
# !pip install beautifulsoup4 requests -q
# !pip install playwright -q && playwright install chromium

import warnings, time, re
import pandas as pd
warnings.filterwarnings("ignore")

# 1) BeautifulSoup — 오늘 핵심 파싱 도구 (대부분 D+010에서 설치됨)
try:
    from bs4 import BeautifulSoup
    HAS_BS4 = True
except ImportError:
    HAS_BS4 = False

# 2) requests — 네트워크 요청용 (없거나 막혀도 폴백으로 진행)
try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False

# 3) Playwright — 브라우저 자동화 (Part 3에서 사용; 없으면 내장 HTML로 폴백)
try:
    import playwright          # 설치 여부만 확인 (실제 import는 함수 안에서)
    HAS_PLAYWRIGHT = True
except ImportError:
    HAS_PLAYWRIGHT = False

print("BeautifulSoup :", HAS_BS4)
print("requests      :", HAS_REQUESTS)
print("Playwright    :", HAS_PLAYWRIGHT, "  (없어도 내장 HTML로 학습 목표 달성)")

In [ ]:
# ─────────────────────────────────────────────
# 모두마켓 신상품 페이지 — 두 버전의 HTML을 노트북 안에 넣어 둡니다(네트워크 불필요).
# ─────────────────────────────────────────────

# (1) 서버가 보내준 그대로 = requests가 받는 것. 상품은 아직 '비어 있음'.
STATIC_HTML = """<!DOCTYPE html>
<html lang="ko">
<head><title>모두마켓 - 신상품</title></head>
<body>
  <h1>모두마켓 신상품</h1>
  <div id="product-list">
    <!-- 상품은 JavaScript가 /api/products 에서 받아와 여기에 그립니다 -->
    <p class="loading">상품을 불러오는 중...</p>
  </div>
  <script src="/static/render_products.js"></script>
</body>
</html>"""

# (2) 브라우저가 JS를 실행해 상품을 다 그린 뒤 = 화면에서 우리가 보는 것.
#     (현장처럼 '오염'을 일부러 심어 둡니다: 공백·결측 카테고리·가격 표기 혼재)
RENDERED_HTML = """<!DOCTYPE html>
<html lang="ko">
<head><title>모두마켓 - 신상품</title></head>
<body>
  <h1>모두마켓 신상품</h1>
  <div id="product-list">
    <div class="product" data-id="P-2001">
      <span class="name">무선 이어폰</span><span class="category">가전</span><span class="price">89,000원</span>
    </div>
    <div class="product" data-id="P-2002">
      <span class="name">  블루투스 스피커  </span><span class="category">가전</span><span class="price">49,000원</span>
    </div>
    <div class="product" data-id="P-2003">
      <span class="name">노트북 거치대</span><span class="price">29,000원</span>
    </div>
    <div class="product" data-id="P-2004">
      <span class="name">기계식 키보드</span><span class="category">가전</span><span class="price">₩119000</span>
    </div>
    <div class="product" data-id="P-2005">
      <span class="name">USB-C 충전기</span><span class="category">가전</span><span class="price">19000</span>
    </div>
    <div class="product" data-id="P-2006">
      <span class="name">보조배터리</span><span class="category">잡화</span><span class="price">39,000원</span>
    </div>
    <div class="product" data-id="P-2007">
      <span class="name">스마트워치</span><span class="category">가전</span><span class="price">159,000원</span>
    </div>
    <div class="product" data-id="P-2008">
      <span class="name">액션캠</span><span class="category">가전</span><span class="price">229,000원</span>
    </div>
  </div>
  <script src="/static/render_products.js"></script>
</body>
</html>"""

print("두 버전 HTML 준비 완료")
print("STATIC_HTML 길이  :", len(STATIC_HTML), "자")
print("RENDERED_HTML 길이:", len(RENDERED_HTML), "자  ← 상품이 들어가 더 깁니다")

## 정적 수집의 실패 재확인 — 왜 결과가 비었나

# 1. 정적 수집의 실패 재확인 — 왜 결과가 비었나

D+010에서 BeautifulSoup으로 정적 페이지를 잘 수집했습니다. 그런데 모두마켓 신상품 페이지에서는 같은 코드가 **빈 결과**를 냈죠. 화면엔 분명 상품이 보이는데도요. 오늘 개념을 이해하려면, 먼저 **그 실패를 정확히 재현하고 원인을 짚는** 것이 출발점입니다.

> ❓ **이 파트에서 답할 질문:** 브라우저 화면엔 상품이 보이는데, `requests`로 받은 HTML에는 왜 상품이 없을까요?

## 💡 쉽게 말하면 — '주문서'와 '완성된 요리'는 다르다

식당에 비유해봅시다. `requests`가 받는 `STATIC_HTML`은 **주문서**입니다. "여기에 상품 목록을 그려 넣어라"라는 *지시*만 적혀 있죠. 실제 **완성된 요리**(상품이 다 그려진 화면)는 주방(브라우저)이 JavaScript를 실행해야 비로소 나옵니다.

```text
서버 → [STATIC_HTML: "상품 불러오는 중..." + 그리라는 지시(script)]  → requests가 받는 것
                                  │
                       브라우저가 script 실행 (JS가 데이터 받아와 그림)
                                  ▼
              [RENDERED_HTML: 상품 8개가 그려진 화면]  → 사람이 보는 것
```

BeautifulSoup은 **받은 HTML 문자열만** 봅니다. JavaScript를 실행하지 못해요. 그래서 주문서(정적)에서 요리(상품)를 찾으니 빈손인 겁니다.

## 자세히 알아보기

| 용어 | 뜻 |
| --- | --- |
| 정적 페이지(static) | HTML 소스 안에 데이터가 **그대로** 들어 있는 페이지 |
| 동적 페이지(dynamic) | JavaScript가 실행돼야 데이터가 **나중에** 채워지는 페이지 |
| 렌더링(rendering) | 브라우저가 HTML+JS를 해석해 **화면을 그리는** 과정 |
| DOM | 브라우저가 메모리에 만든, 렌더링된 문서 구조(Document Object Model) |

> 💡 **핵심:** `requests` + BeautifulSoup는 **렌더링 전(前)** 의 HTML만 봅니다. 동적 페이지의 데이터는 렌더링 후에 생기므로, 정적 도구로는 잡히지 않습니다.

> **읽는 법:** 찾은 상품 수가 **0개**입니다. `#product-list` 안에는 `상품을 불러오는 중...`이라는 안내문만 있죠. 어제 막혔던 장면이 그대로 재현됐습니다. 코드가 틀린 게 아니라, **데이터 자체가 소스에 없었던** 겁니다.

> **읽는 법:** 상품명 글자 자체가 정적 HTML에는 없고 렌더링된 HTML에만 있습니다. **"화면에 보인다 ≠ 소스에 있다"** — 이 한 줄이 정적과 동적을 가르는 결정적 차이입니다.

## 데이터로 확인해 봅시다

- 수집을 시작하기 전, **"이 페이지가 정적인가 동적인가"** 를 먼저 판별해야 도구를 옳게 고릅니다. 정적이면 `requests`+BS로 충분하고, 동적이면 브라우저 자동화(Part 3)가 필요해요.
- 판별의 가장 빠른 방법: 브라우저에서 **"페이지 소스 보기"(View Source)** 로 본 HTML에 데이터가 있으면 정적, 없으면 동적입니다. (소스 보기 = `requests`가 받는 것과 같습니다.)

> 💡 **더 알고 싶다면 (선택):** 동적 페이지라도, 데이터가 `<script type="application/json">`이나 `window.__DATA__ = {...}` 형태로 **소스 안 JSON에 숨어 있는** 경우가 있습니다. 이럴 땐 그 script 태그에서 JSON만 뽑아 쓰면 브라우저 없이도 됩니다. 다만 항상 그렇진 않으니, "소스에 데이터가 있나?"를 먼저 확인하세요.

> 📌 **다른 산업에서는?** 정적·동적 판별은 분야를 가립니다. 금융 시세 대시보드, 부동산 매물 목록, 채용 공고 리스트는 대부분 **동적**으로 그려져 정적 수집이 막힙니다. 반면 공공기관 공지·문서 페이지는 **정적**인 경우가 많아 `requests`만으로 충분합니다.

### 스스로 해보자! ✏️ (1)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `RENDERED_HTML`에 BeautifulSoup을 적용해 `.product`가 몇 개 잡히는지 세어보세요. (정적과 비교!)
2. **(응용)** `#product-list` 안의 글자를 정적·렌더링 각각 출력해, 무엇이 다른지 눈으로 확인하세요.
3. **(생각해보기)** 만약 "소스 보기"에서 데이터가 안 보였다면, 다음에 어떤 도구가 필요할까요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
soup_rendered = BeautifulSoup(RENDERED_HTML, "html.parser")
print("렌더링 HTML에서 찾은 상품 수:", len(soup_rendered.select(".product")))   # 8
```

3번: 데이터가 소스에 없으면 **JavaScript를 실제로 실행해 줄 도구**(브라우저 자동화 = Playwright)가 필요합니다. 바로 다음 Part들의 주제예요.

</details>

### ✅ 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. 정적 페이지와 동적 페이지의 차이를 한 문장으로 말할 수 있나요?
2. BeautifulSoup이 동적 페이지의 데이터를 못 찾는 이유는 무엇인가요?
3. 정적/동적을 빠르게 판별하는 방법은 무엇인가요?

> 💡 **다음 Part 예고:** "JS가 실행돼야 데이터가 생긴다"는 건 알았습니다. 그런데 **정확히 무슨 일이** 벌어지길래 소스와 화면이 달라지는 걸까요? 다음 Part에서 렌더링의 원리를 도식으로 들여다봅니다.

In [ ]:
# 예제: 정적 HTML에서 상품을 찾아보자 — D+010 방식 그대로
soup_static = BeautifulSoup(STATIC_HTML, "html.parser")
products_found = soup_static.select(".product")   # 상품 div 찾기

print("정적 HTML에서 찾은 상품 수:", len(products_found))   # 0
print("#product-list 안의 글자:", repr(soup_static.select_one("#product-list").get_text(strip=True)))

In [ ]:
# 예제: 데이터가 정말 소스에 없는지 직접 검색 — '무선 이어폰'이 STATIC_HTML에 있나?
print("'무선 이어폰'이 정적 HTML에 있나? :", "무선 이어폰" in STATIC_HTML)
print("'무선 이어폰'이 렌더링 HTML에 있나?:", "무선 이어폰" in RENDERED_HTML)
print()
print("→ 화면에 보이는 상품명조차 '서버가 준 HTML'에는 존재하지 않습니다.")

In [ ]:
soup_rendered = BeautifulSoup(RENDERED_HTML, "html.parser")
print("렌더링 HTML에서 찾은 상품 수:", len(soup_rendered.select(".product")))   # 8

## JavaScript 렌더링 원리 — 서버가 준 HTML ≠ 보이는 화면

# 2. JavaScript 렌더링 원리 — 서버가 준 HTML ≠ 보이는 화면

Part 1에서 "정적 HTML엔 데이터가 없다"는 사실을 확인했습니다. 이제 **왜** 그런지, 렌더링이라는 과정에서 무슨 일이 벌어지는지 이해할 차례입니다. 원리를 알면 "이 페이지는 어떤 도구로 수집해야 하는가"가 저절로 보입니다.

> ❓ **이 파트에서 답할 질문:** 서버가 보낸 HTML과 브라우저가 보여주는 화면은, 어떤 과정을 거쳐 달라질까요?

## 💡 쉽게 말하면 — 브라우저가 하는 '추가 작업'

```text
1) 브라우저가 서버에 페이지 요청
        ▼
2) 서버가 STATIC_HTML 응답  ──→  "상품 불러오는 중..." + <script>
        ▼
3) 브라우저가 <script>(JavaScript) 실행
        ▼
4) JS가 /api/products 에서 데이터를 받아와    ←★ 여기서 데이터가 처음 등장
        ▼
5) #product-list 안에 상품 div들을 그림(=DOM 변경)
        ▼
6) 사람이 보는 화면 = RENDERED_HTML (상품 8개)
```

`requests`는 2번에서 멈춥니다(받기만 함). 사람은 6번을 봅니다. **3~5번의 "추가 작업"이 빠진 것** 이 정적 수집이 비는 이유예요.

## 자세히 알아보기

| 용어 | 뜻 | 우리 예시에서 |
| --- | --- | --- |
| 서버 응답 HTML | 서버가 처음 보낸 HTML | `STATIC_HTML` |
| JavaScript 실행 | 브라우저가 script를 돌려 데이터를 받아오고 화면을 바꿈 | 상품 8개 생성 |
| 렌더링된 DOM | JS 실행 후 최종 화면 구조 | `RENDERED_HTML` |
| API 호출 | JS가 데이터를 받아오는 별도 요청 | `/api/products` |

핵심은 이것입니다. **만약 우리가 '렌더링된 DOM'(RENDERED_HTML)을 손에 넣을 수만 있다면, 그 다음은 D+010에서 배운 BeautifulSoup 파싱과 똑같습니다.** 어려운 건 "렌더링된 DOM을 얻는 일"이지, 파싱이 아니에요.

> **읽는 법:** 8개 상품이 표로 잡혔습니다. `select_one(".category")`가 없는 행(P-2003)은 `None`이 들어갔어요 — 수집 단계에서 **결측치가 자연스럽게 생기는** 장면입니다. `price_raw`는 `89,000원`·`₩119000`·`19000`처럼 표기가 제각각이고요. 수집한 데이터가 곧장 깨끗하지 않다는 걸 보여줍니다.

## 수집은 랭글링의 입구 — 배운 기술을 다시 적용

수집한 표에는 오염이 있습니다. 다행히 우리는 이미 **D+003(결측)·D+005(문자열·정규식)** 에서 이걸 다루는 법을 배웠어요. 수집 직후 그대로 재적용하면 됩니다.

> **읽는 법:** 공백이 사라지고(`블루투스 스피커`), 결측 카테고리는 `미분류`로 채워졌으며, `89,000원`·`₩119000`이 모두 정수 가격으로 통일됐습니다. 이제야 평균 가격 같은 계산이 가능하죠. **수집(D+010~011)과 정제(D+002~009)는 하나의 흐름**이라는 걸 직접 체감하는 장면입니다.

> 💡 **개념 연결:** `product_id`·`category`·`price` 컬럼 구성을 눈여겨보세요. 이건 우리가 모듈 내내 써 온 모두마켓 `products` 스키마와 같습니다. 즉 **수집 결과가 다음 모듈(통계)의 입력**으로 자연스럽게 이어집니다.

> 📌 **다른 산업에서는?** "수집 직후 정제"는 어디서나 같습니다. 채용 데이터는 연봉 표기(`5,000만`/`50000000`)를 정수로, 부동산은 면적 단위(`84㎡`/`25평`)를 통일하고, 리뷰 데이터는 공백·이모지·HTML 태그를 제거한 뒤에야 분석에 씁니다.

## 💡 더 알고 싶다면 (선택) — 데이터가 `<script>` 안에 숨어 있을 때

동적 페이지라도, 데이터가 화면에 그려지기 전 **소스의 `<script>` 태그 안 JSON**에 함께 담겨 오는 경우가 있습니다. 이럴 땐 브라우저 없이도 그 JSON만 뽑아 쓸 수 있어요. "동적이면 무조건 Playwright"가 아니라, **소스에 데이터가 있는지 먼저 확인**하는 습관이 이득입니다.

> **읽는 법:** 화면엔 빈 `#app`만 있어도, 소스의 `<script>` 안 JSON에 상품이 들어 있었습니다. 정규식(D+005)으로 그 JSON만 뽑아 곧장 표로 만들었죠. **브라우저 없이 해결되는 경우**라 Playwright보다 빠르고 가볍습니다. 그래서 수집 전 "소스 보기"로 데이터 위치를 먼저 확인하는 게 이득이에요.

### 스스로 해보자! ✏️ (2)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `products_final`에서 `category`가 `가전`인 상품만 골라 개수를 출력하세요.
2. **(응용)** 가격이 가장 비싼 상품의 `name`과 `price`를 출력하세요. (힌트: `sort_values`)
3. **(생각해보기)** 만약 `price` 정제(숫자만 남기기)를 건너뛰고 `"89,000원"` 그대로 두면, `.mean()` 계산은 어떻게 될까요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
가전 = products_final[products_final["category"] == "가전"]
print("가전 상품 수:", len(가전))

top = products_final.sort_values("price", ascending=False).iloc[0]
print("최고가 상품:", top["name"], top["price"], "원")
```

3번: `"89,000원"`은 문자열이라 `.mean()`이 아예 안 되거나(숫자가 아니므로) 오류가 납니다. 그래서 **수집 직후 타입 정리**가 분석의 전제입니다.

</details>

### ✅ 짚고 넘어가기

1. `requests`는 렌더링 과정의 몇 번째 단계에서 멈추나요?
2. "렌더링된 DOM만 얻으면 파싱은 D+010과 같다"는 말의 의미는 무엇인가요?
3. 수집한 데이터에 결측·표기 혼재가 있을 때, 어느 차시의 기술을 다시 쓰나요?

> 💡 **다음 Part 예고:** 결국 관건은 **"렌더링된 DOM을 어떻게 손에 넣느냐"** 입니다. 그러려면 JavaScript를 실제로 실행해 줄 *진짜 브라우저*가 필요해요. 다음 Part에서 브라우저를 코드로 조종하는 **Playwright**를 만납니다.

In [ ]:
# 예제: '렌더링된 DOM'만 있으면, 파싱은 D+010과 똑같다 — 상품을 표로 추출
def parse_products(html):
    """렌더링된 HTML에서 상품을 뽑아 DataFrame으로. (D+010 파싱과 동일한 방식)"""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for item in soup.select(".product"):
        name_el = item.select_one(".name")
        cat_el = item.select_one(".category")     # 없을 수도 있음(결측!)
        price_el = item.select_one(".price")
        rows.append({
            "product_id": item.get("data-id"),
            "name": name_el.get_text() if name_el else None,
            "category": cat_el.get_text(strip=True) if cat_el else None,
            "price_raw": price_el.get_text(strip=True) if price_el else None,
        })
    return pd.DataFrame(rows)

products = parse_products(RENDERED_HTML)
print("수집된 상품 수:", len(products))
display(products)

In [ ]:
# 예제: 수집 데이터에 랭글링 재적용 (D+003 결측 + D+005 문자열/정규식)
clean = products.copy()
clean["name"] = clean["name"].str.strip()                       # D+005: 앞뒤 공백 제거
clean["category"] = clean["category"].fillna("미분류")           # D+003: 결측 대체
clean["price"] = clean["price_raw"].str.replace(r"[^0-9]", "", regex=True).astype(int)  # D+005: 숫자만

products_final = clean[["product_id", "name", "category", "price"]]
display(products_final)
print("category 결측 처리 후:", products_final["category"].value_counts().to_dict())
print("평균 가격:", int(products_final["price"].mean()), "원")

In [ ]:
# 예제: 데이터가 <script> 안 JSON으로 함께 온 경우 — 브라우저 없이 추출 (D+005 정규식 복습)
import json
SCRIPT_JSON_HTML = """<html><body><div id="app"></div>
<script>window.__DATA__ = {"products":[{"name":"무선 이어폰","price":89000},{"name":"액션캠","price":229000}]};</script>
</body></html>"""

m = re.search(r"window\.__DATA__\s*=\s*(\{.*\});", SCRIPT_JSON_HTML)   # script 안 JSON만 뽑기
data = json.loads(m.group(1))
df_from_script = pd.DataFrame(data["products"])
print("화면용 #app 은 비어 있지만, 소스의 script JSON에는 데이터가 있었습니다:")
display(df_from_script)

In [ ]:
# 스스로 해보자! (2)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

가전 = products_final[products_final["category"] == "가전"]
print("가전 상품 수:", len(가전))

top = products_final.sort_values("price", ascending=False).iloc[0]
print("최고가 상품:", top["name"], top["price"], "원")

## Playwright — 브라우저를 코드로 조종하기

# 3. Playwright — 브라우저를 코드로 조종하기

Part 2에서 결론이 났습니다. 동적 페이지를 수집하려면 **JavaScript를 실제로 실행해 렌더링된 DOM을 얻어야** 합니다. 그 일을 해 주는 도구가 **브라우저 자동화(browser automation)** 도구예요. 대표 주자가 **Playwright**입니다.

> ❓ **이 파트에서 답할 질문:** 사람이 손으로 브라우저를 여닫는 대신, 코드로 브라우저를 조종해 렌더링 결과를 얻으려면?

## 💡 쉽게 말하면 — '보이지 않는 브라우저'를 코드가 운전한다

Playwright는 **헤드리스 브라우저(headless browser)** — 화면 없이 background에서 도는 진짜 크롬 — 를 코드로 조종합니다. 사람이 하던 일을 코드가 대신하는 거예요.

```text
사람이 직접:   브라우저 열기 → 주소 입력 → (JS 실행 기다림) → 화면 복사
                              │ 같은 일을
코드(Playwright):  launch() → goto(url) → wait_for_selector() → content()
                                                                   │
                                                          렌더링된 DOM(HTML 문자열) 반환
```

핵심은 `wait_for_selector` — "상품(`.product`)이 화면에 나타날 때까지 기다려라"입니다. JS가 데이터를 그릴 시간을 주는 거죠. 그 뒤 `content()`로 **렌더링된 HTML**을 통째로 가져오면, 나머지는 Part 2의 파싱과 똑같습니다.

## 자세히 알아보기 — Playwright 핵심 API

| 코드 | 하는 일 |
| --- | --- |
| `sync_playwright()` | Playwright 시작 |
| `p.chromium.launch(headless=True)` | 화면 없는 크롬 실행 |
| `browser.new_page()` | 새 탭 열기 |
| `page.goto(url)` | 주소로 이동 |
| `page.wait_for_selector(".product")` | 그 요소가 나타날 때까지 대기 |
| `page.content()` | **렌더링된 DOM** 전체를 HTML 문자열로 |
| `browser.close()` | 브라우저 닫기 |

**설치(한 번만):**

```bash
pip install playwright
playwright install chromium     # 브라우저 실물도 받아야 합니다
```

> ⚠ **주의:** Playwright는 **실제 브라우저를 설치**해야 동작합니다(`playwright install chromium`). 그래서 이 셀은 환경에 따라 못 돌 수 있어요. 아래 코드는 **설치돼 있으면 실행, 없으면 노트북 내장 `RENDERED_HTML`로 자동 폴백**하도록 만들었습니다. 학습 목표(렌더링된 DOM 파싱)는 어느 쪽이든 달성됩니다.

> **읽는 법:** 위 출력이 `폴백(...)`이라면 이 환경엔 Playwright 브라우저가 없는 것입니다(정상입니다!). 그래도 우리는 **저장해 둔 렌더링 결과**를 `rendered`에 담았으므로, 다음 단계 학습엔 지장이 없어요. Playwright가 설치된 환경에서 같은 셀을 돌리면 `playwright(실제 브라우저)`로 표시되고, 진짜 사이트의 렌더링 HTML이 들어옵니다.

> 📸 **실제 실행 화면(참고):** Playwright가 동작하면, 화면 없는 크롬이 다음과 같은 페이지를 메모리에서 렌더링합니다.
> ```text
> ┌─ quotes.toscrape.com/js ──────────────┐
> │  "The world as we have created it ..." │  ← JS로 그려진 인용문
> │   ─ Albert Einstein   (tags: change)   │
> │  "It is our choices ..."               │
> │   ─ J.K. Rowling      (tags: abilities)│
> └────────────────────────────────────────┘
> ```
> 소스 보기(View Source)에는 이 인용문이 **없지만**, Playwright가 렌더링한 `content()`에는 **있습니다.**

> **읽는 법:** 어느 경로든 **"렌더링된 HTML → BeautifulSoup 파싱"** 흐름은 똑같습니다. Playwright의 역할은 딱 하나, *렌더링된 DOM을 가져다주는 것* 뿐이에요. 그 다음은 우리가 D+010에서 이미 익힌 기술입니다. 어려워 보였던 동적 수집이 사실은 "DOM 확보 + 익숙한 파싱"의 조합인 셈이죠.

## 데이터로 확인해 봅시다

- 동적 대시보드(시세·매물·통계 시각화)나 "더 보기" 버튼으로 내용이 늘어나는 페이지는 Playwright 류의 도구가 사실상 필수입니다.
- 다만 브라우저를 띄우는 만큼 **느리고 무겁습니다.** 그래서 "정적으로 되면 정적으로, 정 안 되면 동적으로"가 원칙이에요(도구 선택은 Part 6에서 표로 정리합니다).

> 📌 **다른 산업에서는?** 브라우저 자동화는 수집뿐 아니라 **테스트**에도 쓰입니다. 웹 서비스 회사는 Playwright로 "버튼을 누르면 결제가 되는가"를 자동 점검하고, QA 팀은 화면 회귀 테스트에 활용합니다. 같은 도구, 다른 목적이에요.

정리하는 의미로, **같은 페이지를 두 방식으로 봤을 때**의 차이를 숫자로 비교해봅시다. 정적(requests가 받는 것)과 렌더링(브라우저가 그린 것)의 상품 수를 나란히 보면 차이가 분명합니다.

### 스스로 해보자! ✏️ (3)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `render_with_playwright` 함수에서 **렌더링된 DOM을 가져오는** 한 줄은 무엇인가요? 주석으로 표시해보세요.
2. **(응용)** `wait_for_selector`의 인자를 `.product` 대신 존재하지 않는 `.banana`로 바꾸면 어떤 일이 생길지 예상해보세요. (실제로 바꿔 try/except 메시지를 관찰)
3. **(생각해보기)** 정적 `requests`보다 Playwright가 느린 이유를 한 문장으로 설명해보세요.

<details>
<summary>💡 힌트 (클릭)</summary>

1번: `html = page.content()` 가 렌더링된 DOM 전체를 문자열로 가져오는 줄입니다.

2번: `.banana`는 페이지에 없으므로 `wait_for_selector`가 **타임아웃(시간 초과)** 으로 실패합니다. 셀렉터가 틀리면 "기다려도 안 나타나서" 실패한다는 걸 보여줘요 — Part 4의 '검증'이 중요한 이유입니다.

3번: Playwright는 **진짜 브라우저를 띄우고 JavaScript를 다 실행한 뒤** HTML을 주기 때문에, 받기만 하는 `requests`보다 느리고 메모리도 많이 씁니다.

</details>

### ✅ 짚고 넘어가기

1. Playwright가 동적 수집에서 맡는 *유일한* 핵심 역할은 무엇인가요?
2. `wait_for_selector`는 왜 필요한가요?
3. Playwright는 설치 시 무엇을 추가로 받아야 하나요? (힌트: `playwright install ...`)

> 💡 **다음 Part 예고:** 셀렉터(`.product`)를 손으로 다 알아내는 건 번거롭습니다. 요즘은 **AI에게 수집 코드를 요청**하면 순식간에 초안을 받죠. 그런데 AI가 준 셀렉터가 틀렸다면? 다음 Part에서 **AI를 '검증 가능한 보조'로 쓰는 4단계 워크플로**를 배웁니다.

In [ ]:
# [브라우저 설치 필요: playwright install chromium]
# 동적 페이지를 렌더링해 HTML을 돌려주는 함수 — import는 함수 안에서(미설치 시 호출할 때만 영향)
def render_with_playwright(url, wait_selector, timeout_ms=5000):
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)   # 화면 없는 브라우저
        page = browser.new_page()
        page.goto(url, timeout=timeout_ms)
        page.wait_for_selector(wait_selector, timeout=timeout_ms)  # JS가 그릴 때까지 대기
        html = page.content()                        # 렌더링된 DOM 전체
        browser.close()
    return html

# 실행 시도 — 브라우저/네트워크가 없으면 내장 HTML로 폴백
# (샌드박스: quotes.toscrape.com/js 는 스크래핑 연습 전용으로 공개된 사이트입니다)
try:
    rendered = render_with_playwright("https://quotes.toscrape.com/js/", wait_selector=".quote")
    source_used = "playwright(실제 브라우저)"
except Exception as e:
    rendered = RENDERED_HTML   # 저장해 둔 렌더링 결과로 폴백
    source_used = f"폴백(내장 RENDERED_HTML) — 사유: {type(e).__name__}"

print("렌더링 소스:", source_used)
print("받은 HTML 길이:", len(rendered), "자")

In [ ]:
# 예제: 렌더링된 DOM(rendered)을 파싱 — Part 2의 함수를 그대로 재사용
# 폴백이면 모두마켓 상품 구조이므로 parse_products로, 실제 quotes면 .quote 구조로 분기
soup_r = BeautifulSoup(rendered, "html.parser")

if soup_r.select(".product"):          # 폴백(모두마켓 내장 HTML)인 경우
    result = parse_products(rendered)
    print("파싱 결과(상품):", len(result), "행")
    display(result.head(3))
else:                                   # 실제 quotes 사이트가 렌더링된 경우
    quotes = [q.select_one(".text").get_text(strip=True) for q in soup_r.select(".quote")]
    print("파싱 결과(인용문):", len(quotes), "개")
    for t in quotes[:3]:
        print("-", t[:50], "...")

In [ ]:
# 예제: 같은 페이지, '정적 vs 렌더링'의 수집 결과를 숫자로 비교
n_static = len(BeautifulSoup(STATIC_HTML, "html.parser").select(".product"))
n_rendered = len(BeautifulSoup(RENDERED_HTML, "html.parser").select(".product"))
print(f"requests만 받았을 때 (정적)  -> 상품 {n_static}개")
print(f"렌더링된 DOM을 봤을 때       -> 상품 {n_rendered}개")
print("-> 같은 페이지인데 '언제의 HTML이냐'가 수집 성패를 가릅니다. 그래서 도구 선택이 중요합니다.")

In [ ]:
# 스스로 해보자! (3)
# 아래 주석(#)을 지우고 실험해보세요. (브라우저가 없으면 폴백 메시지가 보입니다)

# try:
#     test = render_with_playwright("https://quotes.toscrape.com/js/", wait_selector=".banana")
#     print("성공:", len(test))
# except Exception as e:
#     print("기다리던 요소가 안 나타나 시간 초과/실패:", type(e).__name__)

## AI 보조 코딩 4단계 — 생성보다 검증

# 4. AI 보조 코딩 4단계 — 생성보다 검증

스크래퍼를 짤 때 셀렉터(`.product .name`)를 일일이 찾는 건 시간이 걸립니다. 그래서 요즘은 AI에게 "이 HTML에서 상품명과 가격을 뽑는 코드를 짜줘"라고 요청하면 초안을 빠르게 받아요. **속도는 AI가, 판단·검증은 사람이** — 이것이 실무의 기본기입니다. 문제는, AI가 준 셀렉터가 **실제 HTML과 다를 수 있다는 점**입니다.

> ❓ **이 파트에서 답할 질문:** AI가 짜준 수집 코드를, 어떻게 *그대로 믿지 않고* 검증하며 쓸까요?

## 💡 쉽게 말하면 — AI 보조 코딩의 4단계

```text
[1] 목적 정의   "무엇을 어떤 형태로?" — 입력(HTML)과 출력(상품명·가격 표)을 한 문장으로
       ↓
[2] AI 코드 생성  명확한 프롬프트로 스크래퍼 초안 요청
       ↓
[3] 검증 ★★★    실제로 실행 → 결과가 비었나? 셀렉터가 맞나? (가장 중요!)
       ↓
[4] 수정        틀린 부분을 사람이 고치고, 윤리 기준(Part 6)도 점검
```

대부분의 사고는 **[3] 검증을 건너뛸 때** 납니다. AI 코드가 그럴듯해 보여 그냥 믿었다가, 빈 결과나 엉뚱한 데이터를 그대로 분석에 쓰는 거죠.

## 자세히 알아보기 — 프롬프트와 검증 체크리스트

> 💬 **AI에게 이렇게 요청해보세요 (프롬프트 예시):**
> *"아래 HTML 문자열이 있어. `class="product"`인 div마다 상품명과 가격이 들어 있어. BeautifulSoup으로 각 상품의 이름과 가격을 뽑아 pandas DataFrame으로 만드는 함수를 짜줘. 셀렉터는 내가 준 HTML을 보고 정확히 써줘."*
> (HTML 구조를 함께 주면 AI가 셀렉터를 더 정확히 맞춥니다.)

> ✅ **이렇게 검증하세요 (검증 체크리스트):**
> 1. **실행되는가?** 오류 없이 도나요?
> 2. **결과 행 수가 0이 아닌가?** 0이면 셀렉터가 틀렸을 가능성 큼.
> 3. **셀렉터가 실제 HTML과 일치하는가?** (`.item`인지 `.product`인지 직접 대조)
> 4. **값이 멀쩡한가?** 가격에 글자가 섞이거나, 이름이 잘리지 않았나?
> 5. **윤리 통과?** 이 페이지를 수집해도 되는가(Part 6)?

이제 "검증을 건너뛰면 어떻게 되는지"를 직접 체험해봅시다.

> **읽는 법:** 결과가 **0행**입니다! AI는 `.item .title` 셀렉터를 썼지만, 실제 HTML은 `.product .name`이었죠. 코드는 **오류 없이** 돌았기에, 검증을 안 했다면 "데이터가 없네?"하고 넘어가거나 빈 표를 분석에 썼을 겁니다. **오류가 안 난다고 맞는 게 아닙니다.** 결과를 눈으로 확인하는 것이 검증의 핵심이에요.

> **읽는 법:** 셀렉터를 실제 구조(`.product`, `.name`)로 고치니 8행이 제대로 잡혔습니다. AI는 **초안의 속도**를 줬고, **맞는지 판단한 건 사람**이에요. 이 4단계(목적→생성→검증→수정)가 AI 시대 데이터 직무의 핵심 역량입니다.

> 📌 **태도 목표:** 이 모듈에서 반복하는 습관 — *"AI 코드를 그대로 믿지 않고 검증한다"* — 는 D+003·D+006·D+009의 *"판단 근거를 기록한다"* 와 같은 결입니다. 도구가 빨라질수록, **검증하고 근거를 남기는 사람의 판단**이 더 중요해집니다.

> 📌 **다른 산업에서는?** "AI 생성 + 사람 검증"은 코드에만 해당하지 않습니다. 마케팅 카피, 금융 리포트 초안, 의료 요약문 모두 AI가 빠르게 만들되 **전문가가 사실·맥락을 검증**한 뒤 씁니다. 검증 없는 자동화는 빠른 실수일 뿐이에요.

검증을 **눈으로만** 하지 말고, 가장 싼 검증(결과가 비었는지)을 **코드로** 못 박을 수도 있습니다. 초안과 수정본에 똑같이 적용해, 통과/실패를 자동으로 표시해봅시다.

### 스스로 해보자! ✏️ (4)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `ai_draft_scraper`가 0행을 낸 이유를 한 줄 주석으로 적어보세요. (어떤 셀렉터가 틀렸나?)
2. **(응용)** `fixed_scraper`에 `category`도 뽑는 줄을 추가해보세요. (없으면 `None` 처리)
3. **(생각해보기)** 검증 체크리스트 5가지 중, 이번 사례에서 문제를 잡아낸 항목은 몇 번이었나요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
"category": cat_el.get_text(strip=True) if cat_el else None,
```

1번: AI는 `.item`과 `.title`을 썼지만 실제 HTML은 `.product`와 `.name`이었습니다.
3번: 체크리스트 **2번(결과 행 수가 0이 아닌가)** 이 문제를 즉시 드러냈습니다. 그래서 "행 수 확인"은 가장 싸고 강력한 검증이에요.

</details>

### ✅ 짚고 넘어가기

1. AI 보조 코딩 4단계를 순서대로 말할 수 있나요?
2. "오류가 안 났는데도 결과가 틀린" 경우가 왜 위험한가요?
3. 검증 체크리스트에서 가장 먼저 확인하기 좋은 항목은 무엇인가요?

> 💡 **다음 Part 예고:** 지금까지는 "페이지를 열어 읽기"만 했습니다. 그런데 로그인을 해야 보이거나, 스크롤해야 더 나오거나, 팝업을 닫아야 하는 페이지는요? 다음 Part에서 **사람의 클릭·입력·스크롤을 코드로 흉내 내는** 법을 봅니다.

In [ ]:
# 예제: AI가 짜줬다고 가정한 '초안' 스크래퍼 — 그런데 셀렉터가 추측이라 틀릴 수 있다
def ai_draft_scraper(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for item in soup.select(".item"):              # ← AI가 추측한 셀렉터 (실제는 .product!)
        name = item.select_one(".title").get_text()
        price = item.select_one(".price").get_text()
        rows.append({"name": name, "price": price})
    return pd.DataFrame(rows)

# [3] 검증 — 그냥 믿지 말고 실행해서 결과를 본다
draft = ai_draft_scraper(RENDERED_HTML)
print("AI 초안 결과 행 수:", len(draft))   # 0 !
print(draft)

In [ ]:
# 예제: [4] 수정 — 실제 HTML을 보고 셀렉터를 바로잡는다
def fixed_scraper(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for item in soup.select(".product"):           # .item → .product 로 수정
        name_el = item.select_one(".name")          # .title → .name 으로 수정
        price_el = item.select_one(".price")
        rows.append({
            "name": name_el.get_text(strip=True) if name_el else None,
            "price": price_el.get_text(strip=True) if price_el else None,
        })
    return pd.DataFrame(rows)

fixed = fixed_scraper(RENDERED_HTML)
print("수정 후 결과 행 수:", len(fixed))   # 8
display(fixed.head())

In [ ]:
# 예제: 검증 체크리스트 2번('결과가 비지 않았나')을 함수로 만들어 적용
def verify_not_empty(df, label):
    ok = len(df) > 0
    print(f"[검증] {label}: {len(df)}행 -> {'통과 OK' if ok else '실패 (셀렉터 의심!)'}")
    return ok

verify_not_empty(ai_draft_scraper(RENDERED_HTML), "AI 초안")
verify_not_empty(fixed_scraper(RENDERED_HTML), "수정본")

In [ ]:
# 스스로 해보자! (4)
# fixed_scraper를 복사해 category를 추가해보세요.

# def fixed_scraper_v2(html):
#     soup = BeautifulSoup(html, "html.parser")
#     rows = []
#     for item in soup.select(".product"):
#         name_el = item.select_one(".name")
#         cat_el = item.select_one(".category")          # 추가
#         price_el = item.select_one(".price")
#         rows.append({
#             "name": name_el.get_text(strip=True) if name_el else None,
#             "category": cat_el.get_text(strip=True) if ___ else None,   # 빈칸: cat_el
#             "price": price_el.get_text(strip=True) if price_el else None,
#         })
#     return pd.DataFrame(rows)
# display(fixed_scraper_v2(RENDERED_HTML))

## 동적 페이지 상호작용 — 로그인·무한스크롤·팝업

# 5. 동적 페이지 상호작용 — 로그인·무한스크롤·팝업

어떤 데이터는 단순히 페이지를 여는 것만으로는 안 보입니다. **로그인** 후에야 보이거나, 아래로 **스크롤**해야 더 불러오거나, **팝업**을 닫아야 하죠. Playwright는 사람이 하는 이런 상호작용을 코드로 흉내 낼 수 있습니다.

> ❓ **이 파트에서 답할 질문:** 사람이 하던 클릭·입력·스크롤을, 어떻게 코드로 대신하게 할까요?

## 💡 쉽게 말하면 — "사람이 하던 동작"을 한 줄씩 코드로

```text
사람의 동작              →   Playwright 코드
아이디/비번 입력              page.fill("input[name=user]", "...")
로그인 버튼 클릭              page.click("button[type=submit]")
요소 나타날 때까지 기다림      page.wait_for_selector(".content")
아래로 스크롤                 page.mouse.wheel(0, 3000)
팝업 닫기                     page.click(".popup-close")
```

## 자세히 알아보기

| 코드 | 하는 일 |
| --- | --- |
| `page.fill(셀렉터, 값)` | 입력창에 글자 채우기 |
| `page.click(셀렉터)` | 버튼·링크 클릭 |
| `page.wait_for_selector(...)` | 요소가 나타날 때까지 대기 |
| `page.mouse.wheel(0, n)` | n픽셀 아래로 스크롤(무한스크롤 로딩 유발) |
| `page.on("dialog", ...)` | 팝업(alert) 자동 처리 |

> ⚠ **중요 — 샌드박스에서만 연습하세요.** 아래 코드들은 **스크래핑이 명시적으로 허용된 연습용 사이트**(`quotes.toscrape.com` 등)를 가정한 *예시*입니다. **실제 상업 사이트(쇼핑몰·SNS)에 로그인해 수집하는 코드는 이 노트북에 넣지 않습니다** — 이용약관 위반·차단·법적 위험 때문이에요. 로그인 우회·CAPTCHA 회피 같은 **탐지 회피 기법도 다루지 않습니다.** 도구의 *동작 방식*을 이해하는 것이 목표입니다.

> **읽는 법:** 세 함수 모두 **정의만** 해 두었습니다(실제 호출 X). 코드를 읽어보면 사람의 동작(입력→클릭→대기→스크롤)이 한 줄씩 대응되는 게 보이죠. 무한스크롤에서 `wait_for_timeout`으로 **로딩을 기다리는 간격**을 둔 점에 주목하세요 — 이건 다음 Part의 "서버에 예의 갖추기"와도 연결됩니다.

> 📌 **다른 산업에서는?** 로그인·스크롤 자동화는 **업무 자동화(RPA)** 와 맞닿아 있습니다. 사내 시스템에 반복 로그인해 리포트를 내려받거나, 정해진 양식을 자동 제출하는 등 — 단, 어디까지나 **권한이 있는 시스템**에 한해서입니다.

### 스스로 해보자! ✏️ (5)

> 정답은 하나가 아닙니다. 일단 읽고 생각해보세요. (이 Part는 실행보다 '판단' 연습입니다.)

1. **(따라하기)** `scrape_with_login`에서 "비밀번호를 입력하는 줄"과 "로그인 후 콘텐츠를 기다리는 줄"을 각각 찾아 주석으로 표시하세요.
2. **(응용)** 무한스크롤에서 `rounds`를 100으로 키우면 서버에 어떤 영향을 줄지 적어보세요.
3. **(생각해보기)** 다음 중 이 노트북에서 **다루지 않기로 한** 것은? (a) 샌드박스 로그인 구조 이해 (b) 실제 쇼핑몰 로그인 우회 (c) 스크롤로 추가 로딩

<details>
<summary>💡 힌트 (클릭)</summary>

2번: `rounds=100`이면 짧은 시간에 페이지를 100번 스크롤하며 **추가 요청을 폭주**시켜 서버에 큰 부담을 줍니다. 차단당하기 쉽고, 예의에도 어긋나요. 필요한 만큼만, 간격을 두고 하는 것이 원칙입니다.

3번: **(b) 실제 쇼핑몰 로그인 우회** — 이용약관 위반·법적 위험이라 다루지 않습니다. (a)와 (c)는 도구의 동작 이해로 OK.

</details>

### ✅ 짚고 넘어가기

1. 사람의 "입력·클릭·스크롤"에 대응하는 Playwright 함수를 하나씩 말할 수 있나요?
2. 무한스크롤에서 스크롤 사이에 대기(`wait_for_timeout`)를 두는 이유는 무엇인가요?
3. 이 노트북이 "샌드박스에서만 연습"하라고 강조하는 이유는 무엇인가요?

> 💡 **다음 Part 예고:** "할 수 있다"와 "해도 된다"는 다릅니다. 스크롤 100번이 서버에 부담을 주듯, 수집에는 **지켜야 할 선**이 있어요. 다음 Part에서 `robots.txt`·요청 예의·저작권·개인정보 — 수집의 윤리와 법을 배웁니다.

In [ ]:
# [브라우저 설치 필요] — '동작 방식'을 보여주는 예시 (샌드박스 quotes.toscrape.com/login 기준)
# 정의만 합니다. 실제 호출은 브라우저가 설치된 환경에서, 허용된 사이트에 한해서만.
def scrape_with_login(url, username, password):
    """로그인이 필요한 페이지 수집의 '구조'를 보여주는 예시."""
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url)
        page.fill("input[name='username']", username)   # 아이디 입력
        page.fill("input[name='password']", password)   # 비번 입력
        page.click("button[type='submit']")             # 로그인 클릭
        page.wait_for_selector(".quote")                # 로그인 후 콘텐츠 대기
        html = page.content()
        browser.close()
    return html

print("로그인 스크래퍼 '정의' 완료 — 호출은 브라우저 설치 + 허용된 사이트에서만.")

In [ ]:
# [브라우저 설치 필요] — 무한스크롤·팝업 처리 '동작 방식' 예시 (정의만)
def scrape_infinite_scroll(url, rounds=3, pause=1.0):
    """아래로 여러 번 스크롤해 추가 로딩을 유발하는 구조."""
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url)
        for _ in range(rounds):
            page.mouse.wheel(0, 4000)     # 아래로 스크롤
            page.wait_for_timeout(int(pause * 1000))  # 새 콘텐츠 로딩 대기(예의 있게)
        html = page.content()
        browser.close()
    return html

def handle_popup(page):
    """팝업(alert/confirm)을 자동으로 닫는 패턴."""
    page.on("dialog", lambda dialog: dialog.dismiss())   # 뜨면 자동 닫기

print("무한스크롤·팝업 처리 함수 '정의' 완료.")

In [ ]:
# 스스로 해보자! (5) — 코드 실행이 아니라, 아래 질문에 주석으로 답해보세요.
# 1) 비번 입력 줄: page.fill("input[name='password']", password)
#    콘텐츠 대기 줄: page.wait_for_selector(".quote")
# 2) rounds=100 이면? → (여기에 적어보세요)
# 3) 다루지 않기로 한 것? → (a/b/c 중 골라 적어보세요)
print("스스로 해보자 (5)는 판단 연습입니다. 위 주석에 답을 적어보세요.")

## 수집 윤리·법·안전 — "가져와도 되는가"를 판단하기

# 6. 수집 윤리·법·안전 — "가져와도 되는가"를 판단하기

지금까지 *어떻게* 수집하는지를 배웠습니다. 그런데 수집은 결국 **"남의 서버에 요청을 보내 데이터를 가져오는"** 행위입니다. 기술만큼 중요한 게 **책임**이에요. "할 수 있다"가 "해도 된다"를 뜻하지 않습니다. 이 Part는 *코드를 짜기 전에 먼저 거쳐야 할 판단*에 관한 것입니다.

> ❓ **이 파트에서 답할 질문:** 어떤 데이터를, 어디까지, 어떻게 수집해도 되는지 무엇을 보고 판단할까요?

## 💡 쉽게 말하면 — 수집 전 체크하는 네 가지

```text
[1] robots.txt   사이트가 "여긴 긁어도/긁지 마"라고 적어둔 규칙
[2] 이용약관·저작권  데이터를 가져가 재배포·상업적 이용해도 되나?
[3] 요청 예의      서버에 부담 주지 않기 (간격 두기, 한꺼번에 X)
[4] 개인정보      이름·연락처 등 개인을 식별하는 데이터는 신중히
```

## 자세히 알아보기

### (1) robots.txt — 사이트의 수집 규칙

대부분의 사이트는 `사이트주소/robots.txt`에 **"어떤 경로를 수집해도 되는지"** 를 적어둡니다. 수집 전에 이걸 확인하는 게 첫 예의이자 기본입니다. 파이썬 표준 라이브러리로 바로 읽을 수 있어요.

> **읽는 법:** `Disallow`에 적힌 경로(`/admin/`, `/private/`, `/cart`)는 `False`(수집 금지), 나머지는 `True`입니다. `Crawl-delay: 1`은 "요청 사이 최소 1초를 두라"는 뜻이에요. **코드를 짜기 전에 이 규칙부터 확인하는 습관**이 핵심입니다. `can_fetch()`가 `False`면, 거기서 멈춰야 합니다.

`robots.txt`는 접근 허용 여부뿐 아니라 **요청 간격(Crawl-delay)** 도 알려줍니다. 코드로 그 값을 읽어 존중합시다.

### (2) 요청 예의 — 서버에 부담 주지 않기

같은 서버에 초당 수백 번 요청하면 서버가 느려지거나 다운될 수 있고, 여러분의 접속이 차단됩니다. 그래서 **요청 사이에 간격을 두고(rate limiting), 자신을 밝히는(User-Agent)** 것이 예의입니다.

> **읽는 법:** `time.sleep`으로 요청 사이에 숨을 돌리고, `User-Agent`로 "나는 학습용 수집기"라고 밝혔습니다. 한꺼번에 몰아치지 않는 것 — 이것이 *차단당하지 않고, 서버에 폐 끼치지 않는* 가장 기본적인 방법입니다.

### (3) 이용약관·저작권, (4) 개인정보

- **이용약관(ToS)·저작권:** robots.txt가 허용해도, **데이터를 재배포·상업적으로 쓰는 것**은 별개 문제입니다. 사이트 약관과 저작권을 따로 확인해야 해요. "수집 가능 ≠ 마음대로 사용 가능".
- **개인정보:** 이름·연락처·주소처럼 **개인을 식별할 수 있는 데이터**는 수집·저장 자체가 법적 위험(개인정보보호법 등)을 동반합니다. 꼭 필요하지 않으면 수집하지 않고, 수집했다면 즉시 비식별화하는 것이 안전합니다.

> 💡 **가장 안전한 길 — 공식 API 먼저.** 많은 서비스가 **공식 API**(데이터를 정해진 규칙으로 제공하는 창구)를 둡니다. API가 있으면 스크래핑보다 API가 1순위예요. 합법적이고, 안정적이며, 차단 걱정도 없습니다.

## 도구·방법 선택 의사결정표

오늘과 어제 배운 것을 한 표로 정리합니다. **코드를 짜기 전에 이 표로 먼저 판단하세요.**

| 상황 신호 | 적합한 방법 | 다룬 곳 |
| --- | --- | --- |
| **공식 API가 있다** | API 사용 (스크래핑 불필요) | 공통 — **1순위 권장** |
| HTML 소스에 데이터가 그대로 있다(정적) | `requests` + BeautifulSoup | D+010 |
| 화면엔 보이는데 소스엔 없다(JS 렌더링) | Playwright | D+011 (오늘) |
| 로그인·무한스크롤·클릭이 필요하다 | Playwright | D+011 (오늘) |
| robots.txt가 `Disallow` | **수집하지 않는다** | 윤리(Part 6) |

> 📌 **태도 목표:** "도구를 고르기 전에 페이지 유형과 **허용 범위**를 먼저 판단한다." 이 한 문장이 오늘 모듈의 핵심 태도이고, D+003·D+006·D+009에서 강조한 *"판단 근거를 기록한다"* 의 연장선입니다.

### 스스로 해보자! ✏️ (6)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** 위 `robots_text`에 `Disallow: /search`를 한 줄 추가한 뒤 다시 파싱해, `/search`가 막히는지 확인하세요.
2. **(응용)** `can_fetch`로 `/products/123`이 수집 가능한지 확인해보세요. (Allow: / 규칙 적용)
3. **(생각해보기)** robots.txt가 허용해도 "가져다 쓰면 안 되는" 경우를 하나 들어보세요. (힌트: 저작권/개인정보)

<details>
<summary>💡 힌트 (클릭)</summary>

```python
robots_text2 = robots_text + "Disallow: /search\n"
rp2 = RobotFileParser()
rp2.parse(robots_text2.splitlines())
print("/search 수집 가능? →", rp2.can_fetch("*", "/search"))        # False
print("/products/123 수집 가능? →", rp2.can_fetch("*", "/products/123"))  # True
```

3번 예시: robots.txt가 페이지 접근을 허용해도, 거기 실린 **기사·사진·가격 DB를 복제해 재배포하거나 상업적으로 파는 것**은 저작권/약관 위반일 수 있습니다. 또 **개인 이름·연락처**가 있는 데이터는 접근 가능 여부와 별개로 수집을 피해야 합니다.

</details>

### ✅ 짚고 넘어가기

1. 수집 전 확인하는 네 가지(robots·약관/저작권·요청 예의·개인정보)를 떠올릴 수 있나요?
2. `robots.txt`의 `Disallow`와 `Crawl-delay`는 각각 무슨 뜻인가요?
3. "수집 가능 ≠ 사용 가능"이 무슨 의미인지 설명할 수 있나요?

> 💡 **다음 단계 예고:** 이제 도구(Playwright)·방법(AI 4단계)·판단(윤리)을 모두 갖췄습니다. 마지막으로, 처음 보는 새 페이지를 **판별 → 수집 → 검증 → 정제 → 윤리 검토**까지 한 흐름으로 해내는 종합 실습으로 넘어갑니다.

# 🧪 종합 실습 — 새 페이지를 책임 있게 수집하기

지금까지 배운 **판별 → 동적 수집 → AI 4단계 검증 → 정제 → 윤리 검토**를 한 흐름에서 모두 써봅니다.

모두마켓이 새로 **"베스트 도서" 추천 페이지**를 열었습니다(가상). 이 페이지도 동적이라, 서버 HTML엔 데이터가 없고 브라우저가 그립니다. 아래 셀을 실행해 두 버전 HTML을 만든 뒤, 세 시나리오를 차례로 해결하고 **수집 리포트**를 작성하세요.

> 💡 순서가 중요합니다. **정적/동적 판별 → 렌더링 DOM 확보 → AI 초안 + 검증 → 정제 → robots/윤리 검토** 순으로 진행합니다.

## 시나리오 1 — 정적/동적 판별

먼저 **이 페이지가 동적인지 판별**합니다(Part 1). 정적 HTML과 렌더링 HTML 각각에서 `.book`이 몇 개 잡히는지 비교하세요.

## 시나리오 2 — AI 초안 + 검증 + 수정

AI에게 "각 `.book`에서 제목·저자·가격을 뽑아줘"라고 요청해 받은 **초안**입니다. 그런데… 늘 그렇듯 검증이 필요합니다. **실행해서 결과 행 수부터** 확인하세요(Part 4 체크리스트 2번).

> **읽는 법:** AI 초안은 또 0행이었습니다(`li.item`이 틀림). 검증으로 잡아내고 `li.book`으로 수정하니 5행이 제대로 잡혔어요. `웹 스크래핑 실전`(978-3)은 `rating`이 없어 `None`입니다 — 수집 단계의 결측이죠.

## 시나리오 3 — 정제 + 모두마켓 스키마로 정리 + 윤리 검토

수집한 표를 **정제(D+003·D+005)** 하고, robots.txt를 **검토**한 뒤, 다음 모듈로 넘길 형태로 정리합니다.

정리된 데이터로 **간단한 분석까지** 해보면 수집의 목적이 분명해집니다. 평점이 있는 도서를 평점 높은 순으로 정렬해봅시다.

> **읽는 법:** 한 줄짜리 HTML이 **판별 → 수집 → 검증 → 정제 → 윤리 검토**를 거쳐, 분석 가능한 깔끔한 표가 됐습니다. `isbn`·`title`·`price`처럼 정돈된 스키마라, 다음 모듈(통계)에서 곧바로 쓸 수 있어요.

## 📊 수집 리포트 작성 (제출물)

지금까지의 과정을 아래 **리포트 양식**에 맞춰 마크다운으로 정리하세요. 이것이 오늘의 최종 산출물이며, 과제로 제출합니다.

```markdown
# 모두마켓 베스트 도서 — 수집 리포트

## 1. 페이지 판별
- 정적/동적: ___ (근거: 소스의 .book 수 ___, 화면의 .book 수 ___)
- 선택한 도구: ___ (이유: ___)

## 2. 수집 (AI 보조 4단계)
- 목적: 각 도서의 제목·저자·가격·평점을 표로
- AI 초안 검증 결과: 행 수 ___ → 문제: 셀렉터 ___ 가 틀림
- 수정: li.item → li.book 등으로 교정, 최종 ___행 수집

## 3. 정제 (재적용한 랭글링)
- 공백 제거: title / 결측 대체: rating(___) / 가격 숫자화: price

## 4. 윤리 검토
- robots.txt: /best-books 수집 가능 ___ , Crawl-delay ___
- 요청 예의: time.sleep + User-Agent 적용 계획
- 저작권/개인정보: (재배포 가능 여부, 개인정보 포함 여부 판단)

## 5. 다음 단계
- 이 데이터로 무엇을 분석할지 (예: 가격-평점 관계)
```

> 💡 빈칸(___)은 위 셀들의 실행 결과를 보고 채우세요. **"왜 그렇게 판단했는지"**(특히 도구 선택·윤리) 한 줄씩 근거를 적으면 훌륭한 리포트가 됩니다.

# ✅ 오늘의 퀴즈

배운 내용을 잠깐 확인해볼게요. 틀려도 괜찮습니다. "이런 걸 배웠지" 하고 떠올리는 용도예요.

### 개념 퀴즈

1. 화면엔 보이는데 페이지 소스엔 데이터가 없습니다. 이 페이지는 정적일까요, 동적일까요? 그 이유는?
2. Playwright가 동적 수집에서 맡는 핵심 역할 한 가지는 무엇인가요?
3. AI 보조 코딩 4단계 중 **가장 중요하고 자주 생략되는** 단계는 무엇인가요?
4. 수집 전에 사이트의 수집 허용 규칙을 확인하는 파일 이름은 무엇인가요?

### 코드 퀴즈

`BEST_RENDERED_HTML`에서 **평점(`.rating`)이 있는 도서만** 골라, 그 권수와 평균 가격을 구하세요. (정제된 `books_final` 활용)

> **읽는 법:** 결측이었던 평점을 `평점없음`으로 채워뒀기에, 그 값을 제외하는 것만으로 "평점 있는 도서"를 깔끔히 골라낼 수 있습니다. 수집(오늘)·결측 처리(D+003)·필터링(D+003)이 한 줄에 모인 장면이에요.

# 🎓 정리 & 다음 시간 예고

## 오늘 배운 것이 어떻게 이어졌나

```text
[Part 1] 정적의 실패          소스엔 데이터가 없다 (화면≠소스)
   ↓  (왜 비었나?)
[Part 2] JS 렌더링 원리       서버 HTML → JS 실행 → 렌더링된 DOM
   ↓  (렌더링된 DOM을 어떻게 얻나?)
[Part 3] Playwright          브라우저를 코드로 조종해 DOM 확보
   ↓  (셀렉터를 빨리 짜려면?)
[Part 4] AI 보조 4단계        생성은 AI, 검증·수정은 사람
   ↓  (로그인·스크롤이 필요하면?)
[Part 5] 동적 상호작용        클릭·입력·스크롤을 코드로 (샌드박스만)
   ↓  (해도 되는가?)
[Part 6] 수집 윤리·법         robots·예의·저작권·개인정보 먼저 판단
   ↓
[종합 실습] 판별→수집→검증→정제→윤리 한 흐름
```

## 한 장 정리표

| 주제 | 핵심 한 줄 | 대표 코드/개념 |
| --- | --- | --- |
| 정적 vs 동적 | 화면에 보인다 ≠ 소스에 있다 | `soup.select(".product")` 0개면 동적 의심 |
| JS 렌더링 | 서버 HTML과 화면은 다르다 | 렌더링된 DOM이 필요 |
| Playwright | 브라우저로 DOM을 가져온다 | `page.content()` |
| AI 4단계 | 생성보다 **검증** | 목적→생성→**검증**→수정 |
| 동적 상호작용 | 사람 동작을 코드로 | `fill`·`click`·`wheel` |
| 윤리 | 가져와도 되는지 먼저 | `robots.txt`·`Crawl-delay`·API 우선 |

## 페이지 신호 → 도구 선택 (판단 지도)

| 페이지에서 본 신호 | 적합한 방법 |
| --- | --- |
| 공식 API가 있다 | **API 우선** (스크래핑 불필요) |
| 소스에 데이터가 그대로 있다 | `requests` + BeautifulSoup (D+010) |
| 화면엔 있는데 소스엔 없다 | Playwright (오늘) |
| 로그인·무한스크롤이 필요하다 | Playwright + 상호작용 (오늘) |
| robots.txt가 Disallow | **수집하지 않는다** |

## 🎓 다음 시간 예고

> **"데이터를 수집하고 정제했으니, 이제 그 데이터로 *무엇을 말할 수 있는지* 검정한다."**
>
> 우리는 D+002~D+009에서 데이터를 *정제*했고, D+010~D+011에서 그 데이터가 *어디서 오는지* 배웠습니다. 이제 데이터가 손에 있으니, 다음 모듈(D+012~)에서는 **통계**로 넘어갑니다. "이 차이가 우연인가, 진짜인가?", "두 변수는 관계가 있는가?" 같은 질문에 **근거를 가지고 답하는** 법을 배웁니다. 수집·정제는 끝이 아니라, 분석의 출발점이었어요.

# 📝 오늘의 과제

오늘 만든 **수집 리포트**를 다듬어 GitHub에 제출합니다.

## 제출물

1. 종합 실습의 베스트 도서 페이지를 수집·정제한 노트북(`.ipynb`)
2. 마무리 양식을 채운 **수집 리포트**(노트북 안 마크다운 셀로 작성)

## 필수 과제

- [ ] 정적/동적을 `.select()` 결과 비교로 판별하고 근거를 적었다.
- [ ] AI 초안의 셀렉터 오류를 **검증으로 찾아** 수정하고, 그 과정을 리포트에 적었다.
- [ ] 수집 데이터에 공백·결측·가격 정제(D+003·D+005)를 재적용했다.
- [ ] `robots.txt`를 `RobotFileParser`로 검토하고, 수집 가능 여부를 판단했다.
- [ ] 마무리 수집 리포트 양식(5개 항목)을 모두 채웠다.

## 심화 과제 (선택)

- [ ] (브라우저 설치 가능 시) `playwright install chromium` 후, 샌드박스 `quotes.toscrape.com/js`를 실제로 렌더링·수집해보고 결과를 비교했다.
- [ ] 수집하려는 가상의 사이트를 하나 정해, **"공식 API가 있는지 → robots.txt → 약관/저작권 → 개인정보"** 순으로 수집 가능 여부를 판단한 윤리 메모를 작성했다. (정답 없음 — 판단 근거가 핵심)

## 제출 방법 (GitHub)

3주차이므로 PR 흐름이 익숙하시죠. 작업 브랜치에서 작업한 뒤 `main`으로 Pull Request를 보냅니다.

```bash
git checkout -b feat/d011-dynamic-scraping
git add D011/
git commit -m "D011 동적 페이지 수집 및 윤리 검토 리포트"
git push origin feat/d011-dynamic-scraping
# 이후 GitHub에서 main으로 PR 생성 → 강사가 줄 단위 코드 리뷰
```

## 평가 기준

| 축 | 무엇을 보나 |
| --- | --- |
| 정확성 | 판별·수집·정제 코드가 오류 없이 돌고 결과가 맞는가 |
| 합리성 | 도구 선택·정제 전략이 페이지 형태에 맞게 선택됐는가 |
| 책임감 | robots.txt·요청 예의·저작권/개인정보를 검토하고 근거를 남겼는가 |

> 💡 모두의연구소의 과제는 순위를 매기지 않습니다. **"어제의 나보다 데이터를 더 책임 있게 다루게 되었는가"** 가 기준이에요. 코드와 판단 근거를 동료와 공유하며 함께 성장합니다.

---

수고하셨습니다! 🎉

오늘 여러분은 "화면엔 보이는데 소스엔 없는" 동적 페이지의 비밀을 풀고, 브라우저를 코드로 조종해 데이터를 가져왔습니다. AI의 초안을 **검증**하며 쓰는 법을 익혔고, 무엇보다 *"가져와도 되는가"를 먼저 묻는* 책임 있는 태도를 배웠어요.

이로써 **데이터 랭글링과 수집** 여정을 마칩니다. 다음 시간부터는 손에 쥔 데이터로 *무엇을 말할 수 있는지* — 통계의 세계로 갑니다. 천천히, 그러나 멈추지 않고 가봅시다. 🚀

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 교안은 생성형 AI를 활용해 제작하고 제작자가 검수했습니다.<br>
무단 복제 및 배포를 금합니다.</sub>

In [ ]:
# 예제: robots.txt 읽고 판단하기 — 네트워크 없이 '내용 파싱'으로 시연(항상 실행)
from urllib.robotparser import RobotFileParser

# 실제 robots.txt가 이렇게 생겼다고 가정합니다.
robots_text = """User-agent: *
Disallow: /admin/
Disallow: /private/
Disallow: /cart
Allow: /
Crawl-delay: 1
"""

rp = RobotFileParser()
rp.parse(robots_text.splitlines())   # 파일 내용을 직접 파싱(네트워크 불필요)

for path in ["/products", "/admin/users", "/private/keys", "/cart"]:
    allowed = rp.can_fetch("*", path)
    print(f"{path:<16} 수집 가능? → {allowed}")

In [ ]:
# [네트워크 필요] 실제 사이트의 robots.txt 확인 — 안 되면 위 로컬 시연으로 충분
try:
    rp_live = RobotFileParser()
    rp_live.set_url("https://books.toscrape.com/robots.txt")
    rp_live.read()                      # 네트워크로 robots.txt 가져오기
    can = rp_live.can_fetch("*", "https://books.toscrape.com/catalogue/")
    print("books.toscrape.com /catalogue 수집 가능? →", can)
except Exception as e:
    print("네트워크 불가 →", type(e).__name__)
    print("→ 괜찮습니다. 위 셀의 로컬 시연으로 robots.txt 판단법은 이미 익혔습니다.")

In [ ]:
# 예제: robots.txt의 Crawl-delay(요청 간격) 읽기
delay = rp.crawl_delay("*")
print("이 사이트가 요구하는 요청 간격(Crawl-delay):", delay, "초")
print(f"-> 그렇다면 요청 사이에 최소 {delay or 1}초는 쉬어주는 것이 예의입니다.")

In [ ]:
# 예제: 요청 사이 간격 두기(rate limiting) 시뮬레이션 — 서버에 예의를 갖추는 패턴
pages = ["/page/1", "/page/2", "/page/3"]
headers = {"User-Agent": "MoodleMarket-Edu-Scraper/1.0 (학습용)"}  # 자신을 밝히기

for page_path in pages:
    # 실제로는 여기서 requests.get(base_url + page_path, headers=headers)
    print(f"요청 → {page_path}  (User-Agent: {headers['User-Agent']})")
    time.sleep(0.5)   # 요청 사이 간격 (robots.txt의 Crawl-delay 존중)

print("\n→ 요청마다 간격을 두고, User-Agent로 신원을 밝혔습니다. 서버 부하를 줄이는 기본 예의입니다.")

In [ ]:
# 스스로 해보자! (6)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

# robots_text2 = robots_text + "Disallow: /search\n"     # 규칙 추가
# rp2 = RobotFileParser()
# rp2.parse(robots_text2.splitlines())
# print("/search 수집 가능? →", rp2.can_fetch("*", "___"))      # 빈칸: "/search"
# print("/products/123 수집 가능? →", rp2.can_fetch("*", "/products/123"))

In [ ]:
# 종합 실습용 페이지 — 두 버전 HTML(네트워크 불필요). 구조가 Part 1~5와 다릅니다(새 셀렉터!).
BEST_STATIC_HTML = """<!DOCTYPE html><html lang="ko"><head><title>모두마켓 베스트 도서</title></head>
<body><h1>이주의 베스트 도서</h1>
  <ul id="book-list"><li class="loading">불러오는 중...</li></ul>
  <script src="/static/render_books.js"></script>
</body></html>"""

BEST_RENDERED_HTML = """<!DOCTYPE html><html lang="ko"><head><title>모두마켓 베스트 도서</title></head>
<body><h1>이주의 베스트 도서</h1>
  <ul id="book-list">
    <li class="book" data-isbn="978-1">
      <h3 class="title">파이썬으로 시작하는 데이터 분석</h3>
      <span class="author">김데이터</span><span class="price">28,000원</span><span class="rating">4.8</span>
    </li>
    <li class="book" data-isbn="978-2">
      <h3 class="title">  모두를 위한 통계학  </h3>
      <span class="author">이통계</span><span class="price">32,000원</span><span class="rating">4.6</span>
    </li>
    <li class="book" data-isbn="978-3">
      <h3 class="title">웹 스크래핑 실전</h3>
      <span class="author">박수집</span><span class="price">₩26000</span>
    </li>
    <li class="book" data-isbn="978-4">
      <h3 class="title">머신러닝 입문</h3>
      <span class="author">최모델</span><span class="price">35000</span><span class="rating">4.9</span>
    </li>
    <li class="book" data-isbn="978-5">
      <h3 class="title">SQL 첫걸음</h3>
      <span class="author">정쿼리</span><span class="price">24,000원</span><span class="rating">4.5</span>
    </li>
  </ul>
  <script src="/static/render_books.js"></script>
</body></html>"""

print("종합 실습용 HTML 준비 완료 (정적/렌더링 2종)")

In [ ]:
# 시나리오 1 — 정적 vs 렌더링 비교로 동적 페이지임을 확인
soup_static_b = BeautifulSoup(BEST_STATIC_HTML, "html.parser")
soup_rendered_b = BeautifulSoup(BEST_RENDERED_HTML, "html.parser")

print("정적 HTML의 .book 수  :", len(soup_static_b.select(".book")))      # 0 → 동적!
print("렌더링 HTML의 .book 수:", len(soup_rendered_b.select(".book")))   # 5
print("\n판별 결과: 소스엔 없고 화면엔 있으므로 → 동적 페이지 (Playwright 필요)")

In [ ]:
# 시나리오 2 — (가정) AI가 준 초안: 셀렉터가 틀렸다
def ai_book_draft(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for li in soup.select("li.item"):          # ← 틀림: 실제는 li.book
        rows.append({"title": li.select_one(".title").get_text()})
    return pd.DataFrame(rows)

print("[검증] AI 초안 결과 행 수:", len(ai_book_draft(BEST_RENDERED_HTML)))   # 0 → 셀렉터 의심!

# 수정: 실제 HTML을 보고 li.book / 올바른 셀렉터로 고친다
def book_scraper(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for li in soup.select("li.book"):
        rating_el = li.select_one(".rating")           # 없을 수 있음(결측!)
        rows.append({
            "isbn": li.get("data-isbn"),
            "title": li.select_one(".title").get_text(),
            "author": li.select_one(".author").get_text(strip=True),
            "price_raw": li.select_one(".price").get_text(strip=True),
            "rating": rating_el.get_text(strip=True) if rating_el else None,
        })
    return pd.DataFrame(rows)

books = book_scraper(BEST_RENDERED_HTML)
print("[수정 후] 행 수:", len(books))
display(books)

In [ ]:
# 시나리오 3 — 정제(공백·결측·가격) + robots.txt 검토
books_clean = books.copy()
books_clean["title"] = books_clean["title"].str.strip()                 # 공백 제거(D+005)
books_clean["rating"] = books_clean["rating"].fillna("평점없음")          # 결측 대체(D+003)
books_clean["price"] = books_clean["price_raw"].str.replace(r"[^0-9]", "", regex=True).astype(int)  # 숫자만(D+005)
books_final = books_clean[["isbn", "title", "author", "price", "rating"]]

print("[정제 완료]")
display(books_final)

# robots.txt 윤리 검토(로컬 시연)
rp_b = RobotFileParser()
rp_b.parse(["User-agent: *", "Allow: /", "Disallow: /admin/", "Crawl-delay: 1"])
print("이 페이지(/best-books) 수집 가능? →", rp_b.can_fetch("*", "/best-books"))
print("평균 도서 가격:", int(books_final["price"].mean()), "원")

In [ ]:
# 종합 — 정리된 데이터로 간단 분석: 평점 있는 도서를 평점 높은 순으로
rated_books = books_final[books_final["rating"] != "평점없음"].copy()
rated_books["rating_num"] = rated_books["rating"].astype(float)
top_books = rated_books.sort_values("rating_num", ascending=False)
print("평점 높은 순 베스트 도서:")
display(top_books[["title", "author", "price", "rating"]])

In [ ]:
# 코드 퀴즈 — 모범 답안
# rating이 '평점없음'이 아닌(=실제 평점이 있는) 도서만 필터
rated = books_final[books_final["rating"] != "평점없음"]

print("평점이 있는 도서 수:", len(rated))
print("평점 있는 도서 평균 가격:", int(rated["price"].mean()), "원")
display(rated[["title", "price", "rating"]])